# Synthetic Single-Cell Proteomics Analysis of Human Cerebellum

**Purpose:** A reproducible, end-to-end computational portfolio project for a simulated single-cell proteomics experiment inspired by published human postmortem brain proteomics workflows.

> **Important:** Every observation in this notebook is simulated. The data do not come from human donors, postmortem tissue, or patients with frontotemporal dementia (FTD). Simulated disease effects are planted to test whether the workflow can recover known signals. This is a methods demonstration, not biological evidence.

## Study design
- 8,124 simulated cells × 3,217 protein features
- 8 simulated donors: 4 Control and 4 FTD
- Five broad cerebellar cell classes: Purkinje neurons, granule neurons, astrocytes, oligodendrocytes and microglia
- Donor-level biological replication; cells are **not** treated as independent biological replicates for differential testing
- Continuous log2-like protein abundance with cell-type effects, donor effects, disease-associated effects and abundance-dependent missingness

## Workflow
1. Generate and save synthetic data
2. Quality control and filtering
3. Per-cell median normalisation
4. Exploratory analysis and PCA/UMAP/Leiden clustering
5. Marker-based cluster annotation
6. Donor-level pseudobulk differential protein abundance within cell types
7. Multiple-testing correction, figures, tables and reproducibility exports

Run the notebook from top to bottom in Google Colab. The first cell installs the required packages.

In [ ]:
# 1. Install dependencies (Google Colab)
%pip -q install scanpy==1.10.4 leidenalg igraph gseapy statsmodels seaborn

In [ ]:
# 2. Imports and reproducibility
import os, sys, json, random, warnings, platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score, silhouette_score

import scanpy as sc
import anndata as ad

warnings.filterwarnings("ignore")
SEED = 20261114
np.random.seed(SEED)
random.seed(SEED)

# Keep outputs and inputs in one project directory.
# Colab uses /content; locally, use the repository root.
if "google.colab" in sys.modules:
    PROJECT_DIR = Path("/content/Single-Cell_Proteomics_DA")
else:
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_DIR / "data" / "synthetic"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results"
FIG_DIR = RESULTS_DIR / "figures"
TABLE_DIR = RESULTS_DIR / "tables"
DE_DIR = RESULTS_DIR / "differential_analysis"

for folder in [DATA_DIR, PROCESSED_DIR, FIG_DIR, TABLE_DIR, DE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Scanpy:", sc.__version__)
print("Project directory:", PROJECT_DIR)

## 3. Generate the synthetic experiment

The simulated abundance matrix is on a log2-like scale. A protein's abundance is influenced by:
- baseline protein abundance;
- cell-type-specific expression;
- donor-to-donor variation;
- a small set of planted FTD-associated effects in selected cell types;
- cell-level technical variation.

Missingness is more likely for low-abundance proteins, mimicking an important feature of mass-spectrometry-based proteomics. The planted effects are recorded in a truth table so that recovery can be evaluated without presenting simulated signals as real discoveries.

In [ ]:
# 3A. Study design
N_CELLS = 8124
N_PROTEINS = 3217
DONORS = [f"D{i:02d}" for i in range(1, 9)]
CONDITION_BY_DONOR = {d: ("Control" if i < 4 else "FTD")
                      for i, d in enumerate(DONORS)}

CELL_TYPES = [
    "Purkinje_neuron",
    "Granule_neuron",
    "Astrocyte",
    "Oligodendrocyte",
    "Microglia"
]
CELL_TYPE_PROPORTIONS = np.array([0.10, 0.30, 0.22, 0.25, 0.13])

# Allocate cells across donors, then assign cell types.
donor_counts = np.full(len(DONORS), N_CELLS // len(DONORS), dtype=int)
donor_counts[:N_CELLS % len(DONORS)] += 1

donor_ids = np.concatenate([
    np.repeat(d, n) for d, n in zip(DONORS, donor_counts)
])
cell_types = np.concatenate([
    np.random.choice(CELL_TYPES, size=n, p=CELL_TYPE_PROPORTIONS)
    for n in donor_counts
])
conditions = np.array([CONDITION_BY_DONOR[d] for d in donor_ids])

# Shuffle cell order so donor blocks are not visually imposed.
perm = np.random.permutation(N_CELLS)
donor_ids, cell_types, conditions = donor_ids[perm], cell_types[perm], conditions[perm]
cell_ids = [f"Cell_{i:05d}" for i in range(N_CELLS)]

metadata = pd.DataFrame({
    "cell_id": cell_ids,
    "donor_id": donor_ids,
    "condition": conditions,
    "cell_type_truth": cell_types
}).set_index("cell_id")

print(metadata.shape)
display(metadata.head())
print("\nCells per condition:")
display(metadata["condition"].value_counts())
print("\nCells per donor:")
display(metadata.groupby(["donor_id", "condition"]).size().unstack())

In [ ]:
# 3B. Create protein names and marker sets
protein_ids = [f"PROT_{i:04d}" for i in range(1, N_PROTEINS + 1)]

marker_sets = {
    "Purkinje_neuron": ["CALB1", "PCP2", "GRID2", "ITPR1", "PVALB", "SYN1", "SNAP25", "RBFOX3"],
    "Granule_neuron": ["GABRA6", "NEUROD1", "ZIC1", "ZIC2", "SLC17A7", "SYN1", "SNAP25", "RBFOX3"],
    "Astrocyte": ["GFAP", "AQP4", "ALDH1L1", "SLC1A3", "S100B", "VIM", "GLUL", "GJA1"],
    "Oligodendrocyte": ["MBP", "PLP1", "MOG", "MAG", "MOBP", "CNP", "SOX10", "CLDN11"],
    "Microglia": ["AIF1", "C1QA", "C1QB", "C1QC", "TYROBP", "P2RY12", "TMEM119", "CSF1R"]
}

# A small planted set of FTD-associated changes for method validation only.
disease_truth = {
    "Astrocyte": {"PROT_0101": 0.85, "PROT_0102": 0.70, "PROT_0103": -0.75},
    "Microglia": {"PROT_0111": 0.90, "PROT_0112": 0.80, "PROT_0113": -0.70},
    "Purkinje_neuron": {"PROT_0121": -0.80, "PROT_0122": 0.75},
    "Granule_neuron": {"PROT_0131": -0.70, "PROT_0132": 0.70},
    "Oligodendrocyte": {"PROT_0141": -0.75, "PROT_0142": 0.75}
}

# Assign interpretable names to marker proteins, with unique IDs retained.
protein_names = {p: p for p in protein_ids}
marker_to_id = {}
marker_counter = 0
for ct, markers in marker_sets.items():
    for marker in markers:
        marker_counter += 1
        pid = protein_ids[marker_counter - 1]
        protein_names[pid] = marker
        marker_to_id[marker] = pid

protein_names_df = pd.DataFrame({
    "protein_id": protein_ids,
    "protein_name": [protein_names[p] for p in protein_ids]
}).set_index("protein_id")

print("Number of proteins:", len(protein_ids))
print("Named marker proteins:", len(marker_to_id))

In [ ]:
# 3C. Simulate log2 protein abundance
# Protein baseline abundance and cell-type-specific effects.
baseline = np.random.normal(loc=20.0, scale=1.8, size=N_PROTEINS)
cell_type_effects = np.random.normal(0, 0.25, size=(len(CELL_TYPES), N_PROTEINS))
ct_to_idx = {ct: i for i, ct in enumerate(CELL_TYPES)}

# Make marker proteins strongly enriched in their expected cell class.
for ct, markers in marker_sets.items():
    for marker in markers:
        pid = marker_to_id[marker]
        pidx = protein_ids.index(pid)
        cell_type_effects[ct_to_idx[ct], pidx] += 2.0

# Donor effects are shared across cells from each donor.
donor_effects = {
    d: np.random.normal(0, 0.18, size=N_PROTEINS) for d in DONORS
}

# Simulated cell-level measurement variation.
X = np.empty((N_CELLS, N_PROTEINS), dtype=np.float32)

for i in range(N_CELLS):
    ct = cell_types[i]
    donor = donor_ids[i]
    row = baseline + cell_type_effects[ct_to_idx[ct]] + donor_effects[donor]
    row = row + np.random.normal(0, 0.35, size=N_PROTEINS)

    # Plant disease effects only in FTD cells of the specified cell type.
    if conditions[i] == "FTD":
        for pid, effect in disease_truth.get(ct, {}).items():
            row[protein_ids.index(pid)] += effect

    X[i, :] = row

# Abundance-dependent missingness: low abundance => greater probability missing.
# This is a simulation choice, not a universal model of proteomics missingness.
scaled = (X - np.nanmean(X)) / np.nanstd(X)
missing_prob = 0.04 + 0.22 / (1 + np.exp(np.clip(scaled, -10, 10)))
missing_mask = np.random.random(X.shape) < missing_prob
X[missing_mask] = np.nan

raw = pd.DataFrame(X, index=metadata.index, columns=protein_ids)
print("Matrix shape:", raw.shape)
print("Overall missingness:", round(raw.isna().mean().mean() * 100, 2), "%")

raw.to_csv(DATA_DIR / "synthetic_raw_log2_abundance.csv.gz", compression="gzip")
metadata.to_csv(DATA_DIR / "cell_metadata.csv")
protein_names_df.to_csv(DATA_DIR / "protein_annotation.csv")

truth_rows = []
for ct, effects in disease_truth.items():
    for pid, effect in effects.items():
        truth_rows.append({"cell_type": ct, "protein_id": pid,
                           "protein_name": protein_names[pid], "true_log2_effect": effect})
truth_df = pd.DataFrame(truth_rows)
truth_df.to_csv(DATA_DIR / "planted_disease_effects_truth.csv", index=False)
display(truth_df)

## 4. Quality control

We assess per-cell observed protein counts and per-protein detection rates. Thresholds are selected transparently for this synthetic dataset and should be adapted to real instrument/platform characteristics. We retain cells with at least 70% of the median observed protein count and proteins detected in at least 50% of retained cells.

In [ ]:
# 4A. QC metrics
cell_qc = pd.DataFrame(index=raw.index)
cell_qc["n_proteins_observed"] = raw.notna().sum(axis=1)
cell_qc["median_observed_intensity"] = raw.median(axis=1)
cell_qc["missing_fraction"] = raw.isna().mean(axis=1)

protein_qc = pd.DataFrame(index=raw.columns)
protein_qc["n_cells_observed"] = raw.notna().sum(axis=0)
protein_qc["detection_fraction"] = raw.notna().mean(axis=0)
protein_qc["mean_intensity"] = raw.mean(axis=0)
protein_qc["variance"] = raw.var(axis=0)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(cell_qc["n_proteins_observed"], bins=40, ax=axes[0])
axes[0].set_title("Observed proteins per cell")
sns.histplot(protein_qc["detection_fraction"], bins=40, ax=axes[1])
axes[1].set_title("Protein detection fraction")
sns.histplot(cell_qc["missing_fraction"], bins=40, ax=axes[2])
axes[2].set_title("Missing fraction per cell")
plt.tight_layout()
plt.savefig(FIG_DIR / "01_qc_distributions.png", dpi=300)
plt.show()

min_cell_proteins = int(0.70 * cell_qc["n_proteins_observed"].median())
keep_cells = cell_qc["n_proteins_observed"] >= min_cell_proteins
raw_qc = raw.loc[keep_cells].copy()
metadata_qc = metadata.loc[keep_cells].copy()

min_detection = 0.50
keep_proteins = raw_qc.notna().mean(axis=0) >= min_detection
raw_qc = raw_qc.loc[:, keep_proteins].copy()

print("Cells retained:", raw_qc.shape[0], "/", raw.shape[0])
print("Proteins retained:", raw_qc.shape[1], "/", raw.shape[1])
print("Post-QC missingness:", round(raw_qc.isna().mean().mean() * 100, 2), "%")

cell_qc.to_csv(TABLE_DIR / "cell_qc_metrics.csv")
protein_qc.to_csv(TABLE_DIR / "protein_qc_metrics.csv")

## 5. Normalisation and embedding matrix

We use per-cell median centring on the log2-like abundance scale. This reduces global cell-level loading/intensity shifts, but assumes that most proteins do not change globally in the same direction. For embedding only, remaining missing values are replaced by each protein's median after normalisation. The imputed matrix is **not** used for differential testing.

In [ ]:
# 5A. Per-cell median normalisation
cell_medians = raw_qc.median(axis=1)
global_median = cell_medians.median()
norm = raw_qc.sub(cell_medians, axis=0).add(global_median)

# 5B. Median imputation for PCA/UMAP/clustering only.
imputer = SimpleImputer(strategy="median")
X_embed = imputer.fit_transform(norm)
print("Embedding matrix:", X_embed.shape)
print("Any missing after imputation:", np.isnan(X_embed).any())

norm.to_csv(PROCESSED_DIR / "normalised_abundance_with_NA.csv.gz", compression="gzip")

## 6. PCA, neighbourhood graph, UMAP and Leiden clustering

We use highly variable proteins to reduce noise and computation. PCA is performed on the imputed, normalised matrix. Scanpy builds a neighbourhood graph, computes UMAP and performs Leiden community detection. These are exploratory clusters and are not automatically equivalent to known biological cell types.

In [ ]:
# 6A. Select variable proteins
protein_var = np.var(X_embed, axis=0)
n_hvg = min(1500, X_embed.shape[1])
hvg_idx = np.argsort(protein_var)[-n_hvg:]
X_hvg = X_embed[:, hvg_idx]

# 6B. Build AnnData object
adata = ad.AnnData(
    X=X_hvg.astype(np.float32),
    obs=metadata_qc.copy(),
    var=pd.DataFrame(index=norm.columns[hvg_idx].astype(str))
)
adata.var["protein_name"] = [protein_names.get(p, p) for p in adata.var_names]

# PCA and graph-based analysis
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=30, svd_solver="arpack", random_state=SEED)
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=25, random_state=SEED)
sc.tl.umap(adata, random_state=SEED)
sc.tl.leiden(adata, resolution=0.6, key_added="leiden_cluster",
             random_state=SEED, flavor="igraph", n_iterations=2)

# Variance explained
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(np.arange(1, len(adata.uns["pca"]["variance_ratio"]) + 1),
        adata.uns["pca"]["variance_ratio"], marker="o")
ax.set(xlabel="Principal component", ylabel="Variance ratio",
       title="PCA variance explained")
plt.tight_layout()
plt.savefig(FIG_DIR / "02_pca_variance.png", dpi=300)
plt.show()

# UMAP coloured by simulated truth and unsupervised clusters.
sc.pl.umap(adata, color=["cell_type_truth", "condition", "donor_id", "leiden_cluster"],
           wspace=0.4, show=False)
plt.savefig(FIG_DIR / "03_umap_overview.png", dpi=300, bbox_inches="tight")
plt.show()

print("ARI: Leiden clusters vs simulated cell labels =",
      round(adjusted_rand_score(adata.obs["cell_type_truth"],
                                adata.obs["leiden_cluster"]), 3))
adata.write_h5ad(PROCESSED_DIR / "proteomics_embedding.h5ad")

## 7. Cluster annotation using protein markers

For this simulated dataset, known marker proteins were planted during data generation. We score clusters using the average abundance of marker panels. This is a transparent demonstration; in real data, marker panels should be selected from appropriate literature and validated in the tissue and assay context. We compare inferred cluster labels with simulated ground truth only as a benchmarking exercise.

In [ ]:
# 7A. Average marker-panel scores per cell
# Use normalised values; marker IDs may not all survive QC, so check availability.
marker_score = pd.DataFrame(index=norm.index)
for ct, markers in marker_sets.items():
    available = [marker_to_id[m] for m in markers
                 if marker_to_id[m] in norm.columns]
    if available:
        marker_score[ct] = norm[available].mean(axis=1)

# Assign marker-score cell type (not a clinical/biological diagnosis).
marker_annotation = marker_score.idxmax(axis=1)
metadata_qc["marker_annotation"] = marker_annotation

# Cluster-level majority annotation and truth comparison.
cluster_map = (
    pd.DataFrame({"cluster": adata.obs["leiden_cluster"].astype(str),
                  "marker_annotation": marker_annotation.loc[adata.obs_names].values,
                  "truth": metadata_qc.loc[adata.obs_names, "cell_type_truth"].values},
                 index=adata.obs_names)
    .groupby("cluster")["marker_annotation"].agg(lambda x: x.value_counts().index[0])
)
adata.obs["cluster_annotation"] = adata.obs["leiden_cluster"].astype(str).map(cluster_map).astype("category")

print("Cluster-to-marker annotation:")
display(cluster_map.to_frame("predicted_cell_type"))

print("Marker annotation accuracy against simulated labels:",
      round((marker_annotation == metadata_qc["cell_type_truth"]).mean(), 3))

sc.pl.umap(adata, color=["cluster_annotation", "cell_type_truth"],
           wspace=0.4, show=False)
plt.savefig(FIG_DIR / "04_umap_cell_type_annotation.png", dpi=300, bbox_inches="tight")
plt.show()

pd.crosstab(metadata_qc["cell_type_truth"], marker_annotation).to_csv(
    TABLE_DIR / "marker_annotation_confusion_matrix.csv"
)

## 8. Differential protein abundance using donor-level pseudobulk

**Critical design point:** cells from the same donor are not independent biological replicates. We therefore aggregate observed normalised protein values within each donor × cell type and compare the four donor-level pseudobulk profiles per condition. For each cell type and protein, we use a Welch two-sample t-test on donor-level means and Benjamini–Hochberg correction within each cell type.

This is a simple, transparent demonstration. With real data, consider models that account for donor, batch, covariates and missingness, and ensure adequate independent donors. The small simulated study is not powered to establish real disease biology.

In [ ]:
# 8A. Pseudobulk: mean observed normalised abundance per donor and cell type.
# No embedding imputation is used here.
long_meta = metadata_qc[["donor_id", "condition", "cell_type_truth"]].copy()
long_meta = long_meta.rename(columns={"cell_type_truth": "cell_type"})

pseudobulk = {}
for ct in CELL_TYPES:
    cells_ct = long_meta.index[long_meta["cell_type"] == ct]
    tmp = norm.loc[cells_ct].join(long_meta.loc[cells_ct])
    pb = tmp.groupby(["donor_id", "condition"])[norm.columns].mean()
    pseudobulk[ct] = pb

# Welch t-tests with BH correction, per cell type.
from scipy.stats import ttest_ind

de_results = []
for ct, pb in pseudobulk.items():
    # Separate donor-level rows by condition.
    ctrl = pb.xs("Control", level="condition")
    ftd = pb.xs("FTD", level="condition")

    for protein in norm.columns:
        a = ctrl[protein].dropna().values
        b = ftd[protein].dropna().values
        if len(a) < 2 or len(b) < 2:
            continue
        stat, pval = ttest_ind(b, a, equal_var=False, nan_policy="omit")
        de_results.append({
            "cell_type": ct,
            "protein_id": protein,
            "protein_name": protein_names.get(protein, protein),
            "mean_control": np.mean(a),
            "mean_ftd": np.mean(b),
            "log2_effect_ftd_minus_control": np.mean(b) - np.mean(a),
            "p_value": pval
        })

de = pd.DataFrame(de_results)
de["padj"] = np.nan
for ct in CELL_TYPES:
    idx = de.index[de["cell_type"] == ct]
    if len(idx):
        de.loc[idx, "padj"] = multipletests(de.loc[idx, "p_value"], method="fdr_bh")[1]

de = de.sort_values(["cell_type", "padj", "p_value"])
de.to_csv(DE_DIR / "all_cell_type_differential_proteins.csv", index=False)
display(de.head(15))

In [ ]:
# 8B. Volcano plots for each cell type
for ct in CELL_TYPES:
    sub = de[de["cell_type"] == ct].copy()
    sub["neglog10_padj"] = -np.log10(sub["padj"].clip(lower=1e-300))
    sig = (sub["padj"] < 0.05) & (sub["log2_effect_ftd_minus_control"].abs() >= 0.5)

    plt.figure(figsize=(7, 5))
    plt.scatter(sub["log2_effect_ftd_minus_control"], sub["neglog10_padj"],
                s=10, alpha=0.55)
    plt.scatter(sub.loc[sig, "log2_effect_ftd_minus_control"],
                sub.loc[sig, "neglog10_padj"], s=16, alpha=0.85)
    plt.axvline(-0.5, linestyle="--", linewidth=1)
    plt.axvline(0.5, linestyle="--", linewidth=1)
    plt.axhline(-np.log10(0.05), linestyle="--", linewidth=1)
    plt.xlabel("Mean log2 abundance difference (FTD - Control)")
    plt.ylabel("-log10 adjusted p-value")
    plt.title(f"Synthetic differential abundance: {ct}")
    plt.tight_layout()
    safe_ct = ct.replace(" ", "_")
    plt.savefig(FIG_DIR / f"volcano_{safe_ct}.png", dpi=300)
    plt.show()

    sub.loc[sig].to_csv(DE_DIR / f"significant_{safe_ct}.csv", index=False)

print("Number of significant proteins by cell type (FDR < 0.05):")
display(de.assign(significant=de["padj"] < 0.05)
        .groupby("cell_type")["significant"].sum().to_frame("n_significant"))

## 9. Benchmark recovery of planted synthetic effects

Because this is simulated data, we know which proteins were deliberately altered. We can compare the differential analysis results with the planted truth table. This is a software/workflow validation exercise only; it is not evidence that these proteins are associated with FTD in real patients.

In [ ]:
# 9A. Match planted effects to differential results
benchmark = truth_df.merge(
    de,
    on=["cell_type", "protein_id", "protein_name"],
    how="left"
)
benchmark["direction_recovered"] = (
    np.sign(benchmark["true_log2_effect"]) ==
    np.sign(benchmark["log2_effect_ftd_minus_control"])
)
benchmark["fdr_significant"] = benchmark["padj"] < 0.05
benchmark.to_csv(TABLE_DIR / "planted_effect_recovery_benchmark.csv", index=False)
display(benchmark)

print("Planted effects recovered in correct direction:",
      round(benchmark["direction_recovered"].mean(), 3))
print("Planted effects significant at FDR < 0.05:",
      int(benchmark["fdr_significant"].sum()), "/", len(benchmark))

## 10. Functional enrichment (optional)

The synthetic protein identifiers are not real gene/protein accessions, so they cannot be meaningfully submitted to pathway databases. We therefore do not run biological enrichment on arbitrary simulated IDs. In a real project, map protein identifiers to gene symbols/UniProt accessions and perform enrichment with a suitable background universe. The example below is a placeholder showing where a real enrichment step belongs.

In [ ]:
# Optional real-data extension:
# 1. Map significant protein IDs to reviewed UniProt accessions / gene symbols.
# 2. Select a biologically justified background: all proteins tested in that cell type.
# 3. Run GO / Reactome / pathway enrichment (e.g. gseapy) and report database/version.
#
# Do NOT submit PROT_0001-style synthetic identifiers to enrichment databases.

## 11. Export a compact results summary and reproducibility metadata

In [ ]:
# 11A. Summary metrics
summary = {
    "project": "Synthetic single-cell proteomics analysis",
    "seed": SEED,
    "initial_cells": int(raw.shape[0]),
    "initial_proteins": int(raw.shape[1]),
    "cells_after_qc": int(raw_qc.shape[0]),
    "proteins_after_qc": int(raw_qc.shape[1]),
    "raw_missing_fraction": float(raw.isna().mean().mean()),
    "post_qc_missing_fraction": float(raw_qc.isna().mean().mean()),
    "donors": int(metadata_qc["donor_id"].nunique()),
    "conditions": metadata_qc["condition"].value_counts().to_dict(),
    "cell_types": metadata_qc["cell_type_truth"].value_counts().to_dict(),
    "leiden_clusters": int(adata.obs["leiden_cluster"].nunique()),
    "ari_leiden_vs_truth": float(adjusted_rand_score(
        adata.obs["cell_type_truth"], adata.obs["leiden_cluster"])),
    "marker_annotation_accuracy_vs_truth": float(
        (marker_annotation == metadata_qc["cell_type_truth"]).mean())
}
with open(RESULTS_DIR / "analysis_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# Save metadata and analysis objects.
metadata_qc.to_csv(PROCESSED_DIR / "cell_metadata_after_qc.csv")
adata.obs.to_csv(TABLE_DIR / "cell_embeddings_and_clusters.csv")
pd.DataFrame(adata.obsm["X_pca"], index=adata.obs_names).to_csv(
    TABLE_DIR / "pca_coordinates.csv")
pd.DataFrame(adata.obsm["X_umap"], index=adata.obs_names,
             columns=["UMAP1", "UMAP2"]).to_csv(TABLE_DIR / "umap_coordinates.csv")

print(json.dumps(summary, indent=2))
print("\nResults saved under:", RESULTS_DIR)

## 12. Interpretation and limitations

### What this notebook demonstrates
- Construction of a controlled synthetic single-cell proteomics dataset
- Cell/protein quality control and missingness assessment
- Per-cell normalisation
- PCA, UMAP and Leiden clustering
- Marker-panel cell-type annotation
- Donor-level pseudobulk differential protein abundance
- Multiple-testing correction and recovery of planted effects
- Reproducible exports for GitHub

### Limitations and cautions
1. **Synthetic data:** no conclusions about human cerebellum, FTD or any protein can be drawn.
2. **Simplified measurement model:** real proteomics data may have batch effects, run-order effects, peptide-to-protein summarisation issues, complex missingness and platform-specific biases.
3. **Cell-type annotation:** marker panels are illustrative and should be validated for the tissue and assay.
4. **Differential testing:** four donors per group is a small demonstration. Real studies need power calculations, covariate-aware models and adequate biological replication.
5. **Pseudobulk:** donor-level aggregation avoids treating thousands of cells from a few donors as independent replicates, but the simple Welch test is not a substitute for a carefully specified study model.
6. **Spatial proteomics:** this notebook does not model spatial coordinates or spatial neighbourhoods. That would be a separate extension.
7. **FTD biology:** FTD is heterogeneous. Real analyses should consider clinical/genetic subtype, pathology, brain region, cell composition, postmortem interval, age, sex and technical covariates where available.

### Suggested next extensions
- Add batch and run-order effects, then evaluate correction strategies.
- Compare alternative normalisation methods.
- Evaluate clustering stability across resolutions and random seeds.
- Add a spatial coordinate layer and spatial neighbourhood analysis.
- Replace synthetic features with real protein accessions when analysing a suitable public dataset.

**Reference inspiration:** Davis, S., Scott, C., Ansorge, O., & Fischer, R. (2019). Development of a Sensitive, Scalable Method for Spatial, Cell-Type-Resolved Proteomics of the Human Brain. Journal of Proteome Research, 18(4), 1787–1795. https://doi.org/10.1021/acs.jproteome.8b00981

**Reproducibility:** random seed = `20261114`. The synthetic dataset and all outputs are regenerated by running this notebook from top to bottom.